In [1]:
import sys

sys.path.append("..")

In [2]:
import os
from src.api.dependencies.injectables import (
    get_mongo_vdb,
    get_chat_model,
    get_topic_selector,
    get_rag_engine,
    get_topic_prompt_builder,
    get_embedding_model,
    get_agent,
    get_election_searcher,
)
from src.mongo import get_mongo_db
from src import ENV
from src.agent.schemas import Platform, Topic
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from IPython.display import display, Markdown
from langchain_core.messages import AIMessage, HumanMessage


In [3]:
chat_model = get_chat_model()
emb_model = get_embedding_model()
topic_selector = get_topic_selector()
db = get_mongo_db()
vector_db = get_mongo_vdb(emb_model, db)
rag_engine = get_rag_engine(vector_db)
search_election = get_election_searcher(chat_model, vector_db)
topic_prompt_builder = get_topic_prompt_builder(db, vector_db, search_election)

In [4]:
agent = get_agent(
    chat_model=chat_model,
    classify_topic=topic_selector,
    rag_retrieve=rag_engine,
    build_topic_prompts=topic_prompt_builder,
)  # type: ignore

In [5]:
history = []
query = ""
result = ""

In [8]:
if query and result:
    history.extend([HumanMessage(query), AIMessage(result)])
query = "Que va a pasar en el mes de octubre?"
result = ""
async for token in agent.stream([*history, HumanMessage(content=query)]):
    if token.type == "info" or token.type == "error":
        display(Markdown(token.content), clear=True)
        continue
    result += token.content
    display(Markdown(result), clear=True)
# display(Markdown(result), clear=False)

Según la información encontrada, en octubre de 2025 se llevará a cabo la segunda vuelta electoral en Bolivia, que está programada para el domingo 19 de octubre. Durante ese mes, también estarán en vigor varias prohibiciones relacionadas con manifestaciones públicas, consumo de alcohol, portación de armas, actos públicos, traslado de electores y circulación de vehículos, entre otras restricciones. Además, el Sistema de Resultados Electorales Preliminares (Sirepre) dará resultados preliminares el día de la elección, con el fin de informar a la ciudadanía y medios de comunicación, aunque estos resultados no son vinculantes ni definitivos.  

Para más detalles, puedes consultar: [Sistema de Resultados Electorales Preliminares (Sirepre)](https://www.oep.org.bo).

In [7]:
%rm -rf chroma_db
%cp -r ../chroma_db .
